# Standard MoE vs DeepSeek MoE: A Practical Comparison

So welcome to this notebook! In this notebook, we are gonna compare two different Mixture-of-Experts (MoE) architectures: the standard MoE approach used in many large language models versus the innovative DeepSeek-MoE architecture. We'll build both from scratch, train them on the same dataset, and see how they stack up against each other.

## What Are We Comparing?

**Standard MoE Architecture:**
- The "classic" approach used in models like Switch Transformers and GShard
- All experts are equal in size and function
- Uses an auxiliary loss to balance expert utilization
- Top-K routing sends each token to K experts

**DeepSeek MoE Architecture:**
- A newer approach with a hybrid expert system
- Combines "shared experts" (everyone uses them) with "routed experts" (only specific tokens use them)
- Uses a bias-only routing mechanism instead of auxiliary loss
- Aims for better quality and efficiency

## What Will We Learn?

By the end of this notebook, we'll understand:
1. How each architecture is implemented
2. Which one trains faster
3. Which one achieves better results
4. How experts are utilized in each approach

**Important Note:** The models we're building here are relatively small (only 6 layers with 512 embedding dimension) for demonstration purposes, so the differences won't be as dramatic as they would be at scale. In production-scale models with billions of parameters, these architectural differences would have much larger impacts on training efficiency and model quality.

In [12]:
!pip install datasets tiktoken tqdm numpy matplotlib pandas seaborn -q


[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python -m pip install --upgrade pip


## Setup and Environment Preparation

We'll start by installing the necessary dependencies for our experiment. This includes:
- `datasets`: To load and process the TinyStories dataset
- `tiktoken`: For tokenization using OpenAI's tokenizer (compatible with GPT-2)
- `tqdm`: For progress tracking during data processing and training
- `numpy`, `matplotlib`, `pandas`, `seaborn`: For data manipulation and visualization

These dependencies enable us to efficiently train our models and analyze the results in a systematic way.

In [13]:
# =============================================================================
# STAGE 1: SETUP, CONFIGURATION, AND DATA PREPARATION
# =============================================================================

# 1.1. Import Core Libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import os
import time
from contextlib import nullcontext
from dataclasses import dataclass

# 1.2. Import Helper Libraries
import numpy as np
from datasets import load_dataset
import tiktoken
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import pandas as pd

print("All dependencies imported successfully.")

All dependencies imported successfully.


## Stage 1: Setup, Configuration, and Data Preparation

In this first stage, we'll:
1. Import necessary libraries for deep learning (PyTorch), data processing, and visualization
2. Configure the training parameters, including batch size, context length, learning rate, etc.
3. Define the DeepSeekMoEConfig class, which contains the architecture specifications for both models
4. Prepare and tokenize the TinyStories dataset

The DeepSeek MoE architecture has some key hyperparameters:
- `moe_n_routed_experts`: Number of specialized experts that process specific tokens (16 by default)
- `moe_top_k`: Number of experts selected per token (2 by default)
- `ds_moe_n_shared_experts`: Number of shared experts that process all tokens (2 by default)
- `ds_moe_shared_expert_hidden_dim`: Dimensionality of the shared experts (larger than routed experts)

For the standard MoE baseline, we'll adjust the number of experts to ensure both models have comparable parameter counts.

In [14]:
# 1.3. Global Training Configuration
BATCH_SIZE = 32
BLOCK_SIZE = 256
MAX_ITERS = 5000
EVAL_INTERVAL = 250
EVAL_ITERS = 100
LEARNING_RATE = 3e-4
WARMUP_ITERS = 200
MIN_LR = 3e-5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'
PTDTYPE = {'float16': torch.float16, 'bfloat16': torch.bfloat16, 'float32': torch.float32}[DTYPE]
CTX = nullcontext() if DEVICE == 'cpu' else torch.amp.autocast(device_type=DEVICE, dtype=PTDTYPE)
SCALER = torch.cuda.amp.GradScaler(enabled=(DTYPE == 'float16'))

/tmp/ipykernel_279/1440405301.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  SCALER = torch.cuda.amp.GradScaler(enabled=(DTYPE == 'float16'))


### Global Training Configuration

Here we define the key training parameters that will be used for both models:

- `BATCH_SIZE=32`: Number of sequences processed in parallel
- `BLOCK_SIZE=256`: Context length (number of tokens per sequence)
- `MAX_ITERS=5000`: Total number of training iterations
- `EVAL_INTERVAL=250`: How often to evaluate on the validation set
- `EVAL_ITERS=100`: Number of batches to use for evaluation
- `LEARNING_RATE=3e-4`: Initial learning rate
- `WARMUP_ITERS=200`: Number of iterations for learning rate warmup
- `MIN_LR=3e-5`: Minimum learning rate after decay

We also set up the device configuration (CUDA/CPU) and mixed precision training to optimize performance. Using `bfloat16` or `float16` significantly improves training speed on compatible GPUs while maintaining numerical stability.

In [15]:
@dataclass
class DeepSeekMoEConfig:
    # Core Transformer Params
    n_layer: int = 6
    n_head: int = 8
    n_embd: int = 512
    vocab_size: int = 50257
    block_size: int = BLOCK_SIZE
    dropout: float = 0.1
    bias: bool = False

    # MoE Specific Params for Fine-Grained Expert Segmentation
    moe_n_routed_experts: int = 16
    moe_top_k: int = 2
    moe_expert_hidden_dim: int = 512 # (n_embd * 4) / 4

    # Shared Expert Isolation
    ds_moe_n_shared_experts: int = 2
    ds_moe_shared_expert_hidden_dim: int = 1024 # Larger for generalist role

MODEL_CONFIG = DeepSeekMoEConfig()

print("\n--- Notebook Configuration ---")
print(f"Device: {DEVICE} with dtype: {DTYPE}")
print(f"Model: {MODEL_CONFIG.n_layer} layers, {MODEL_CONFIG.n_head} heads, {MODEL_CONFIG.n_embd} embedding dim.")
print(f"MoE: {MODEL_CONFIG.moe_n_routed_experts} routed experts (top {MODEL_CONFIG.moe_top_k}) + {MODEL_CONFIG.ds_moe_n_shared_experts} shared experts.")
print("-" * 30)


--- Notebook Configuration ---
Device: cuda with dtype: bfloat16
Model: 6 layers, 8 heads, 512 embedding dim.
MoE: 16 routed experts (top 2) + 2 shared experts.
------------------------------


### DeepSeek MoE Configuration

This is where we define the core architecture of our DeepSeek MoE model. The configuration includes:

**Core Transformer Parameters:**
- 6 layers with 8 attention heads each
- 512 embedding dimension (relatively small for demo purposes)
- Context length of 256 tokens

**MoE-Specific Parameters:**
- 16 routed experts (specialists) with top-2 routing
- Expert hidden dimension of 512 (reduced from the typical 4x expansion)

**Shared Expert Parameters:**
- 2 shared experts (generalists)
- Shared expert hidden dimension of 1024 (2x larger than routed experts)

This configuration represents the key innovation in DeepSeek's approach: having a small number of larger shared experts that process all tokens, combined with many smaller specialized experts that only process specific tokens through routing.

In [16]:
# 1.5. Dataset and Tokenizer Configuration
DATASET_NAME = "roneneldan/TinyStories"
TRAIN_BIN_FILE = "train.bin"
VAL_BIN_FILE = "val.bin"
TOKENIZER_MODEL = "gpt2"

### Dataset and Tokenization Setup

For our experiments, we'll use the **TinyStories** dataset, which contains simple stories suitable for language modeling tasks. We'll tokenize the data using the GPT-2 tokenizer from the `tiktoken` library.

The process involves:
1. Loading the dataset using the Hugging Face `datasets` library
2. Tokenizing each example with the GPT-2 tokenizer 
3. Adding an end-of-text token to each story
4. Writing the tokenized data to memory-mapped binary files for efficient loading during training

This approach allows us to process the data once and then efficiently load random batches during training, which is crucial for training language models effectively.

In [17]:
# 1.6. Tokenization and File Writing
if not (os.path.exists(TRAIN_BIN_FILE) and os.path.exists(VAL_BIN_FILE)):
    print("\nBinary data files not found. Starting data download and tokenization...")
    ds = load_dataset(DATASET_NAME)
    enc = tiktoken.get_encoding(TOKENIZER_MODEL)

    def process_and_tokenize(example):
        ids = enc.encode_ordinary(example['text'])
        # Add EOS token to signify document end
        ids.append(enc.eot_token)
        return {'ids': ids, 'len': len(ids)}

    tokenized_ds = ds.map(
        process_and_tokenize,
        remove_columns=['text'],
        desc="Tokenizing splits",
        num_proc=os.cpu_count(),
    )

    for split, dset in tokenized_ds.items():
        split_name = 'val' if split == 'validation' else split
        filename = f"{split_name}.bin"
        arr = np.memmap(filename, dtype=np.uint16, mode='w+', shape=(sum(dset['len']),))
        total_tokens = len(arr)
        print(f"Writing {total_tokens:,} tokens to {filename}...")
        
        idx = 0
        for example in tqdm(dset, desc=f"Writing {split_name} data"):
            arr[idx : idx + example['len']] = example['ids']
            idx += example['len']
        arr.flush()
    print("\nTokenization and file writing complete.")
else:
    print("\nFound existing train.bin and val.bin files. Skipping data preparation.")


Found existing train.bin and val.bin files. Skipping data preparation.


In [18]:
# 1.7. Data Loading Function
def get_batch(split):
    filename = TRAIN_BIN_FILE if split == 'train' else VAL_BIN_FILE
    data = np.memmap(filename, dtype=np.uint16, mode='r')
    ix = torch.randint(len(data) - BLOCK_SIZE, (BATCH_SIZE,))
    x = torch.stack([torch.from_numpy((data[i:i+BLOCK_SIZE]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+BLOCK_SIZE]).astype(np.int64)) for i in ix])
    return x.to(DEVICE), y.to(DEVICE)

print("\nStage 1 Complete: Setup and data utilities are ready.")
print("=" * 60)


Stage 1 Complete: Setup and data utilities are ready.


In [19]:
# =============================================================================
# STAGE 2: CONSTRUCTING THE CORE ARCHITECTURAL LAYERS
# =============================================================================

# -----------------------------------------------------------------------------
# 2.1. Multi-Head Attention
# -----------------------------------------------------------------------------

class MultiHeadAttention(nn.Module):
    """
    Implements the Multi-Head Self-Attention mechanism as described in the
    "Attention is All You Need" paper.
    """
    def __init__(self, config: DeepSeekMoEConfig):
        super().__init__()
        assert config.n_embd % config.n_head == 0, "Embedding dimension must be divisible by number of heads"

        # Key, Query, Value projections for all heads, but in a single batch operation
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # Output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        # Regularization
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout
        
        # Flash Attention-based scaled dot product attention is only available for PyTorch >= 2.0
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
        if not self.flash:
            print("WARNING: Using manual attention implementation. For faster training, upgrade PyTorch to 2.0 or higher.")
            # Causal mask to ensure that attention is only applied to the left in the input sequence
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                        .view(1, 1, config.block_size, config.block_size))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.size() # Batch size, sequence length, embedding dimensionality (n_embd)

        # Calculate query, key, values for all heads in batch and move head forward to be the batch dim
        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)
        
        # Reshape for multi-head attention: (B, T, C) -> (B, n_head, T, head_size)
        head_size = C // self.n_head
        k = k.view(B, T, self.n_head, head_size).transpose(1, 2)
        q = q.view(B, T, self.n_head, head_size).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_size).transpose(1, 2)

        # Causal self-attention; either with Flash Attention or manual implementation
        if self.flash:
            # Efficient attention using Flash Attention kernel
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0, is_causal=True)
        else:
            # Manual implementation of attention
            # (B, n_head, T, head_size) @ (B, n_head, head_size, T) -> (B, n_head, T, T)
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            # (B, n_head, T, T) @ (B, n_head, T, head_size) -> (B, n_head, T, head_size)
            y = att @ v

        # Re-assemble all head outputs side by side: (B, n_head, T, head_size) -> (B, T, C)
        y = y.transpose(1, 2).contiguous().view(B, T, C)

        # Output projection
        y = self.resid_dropout(self.c_proj(y))
        return y

# --- Verification Step ---
print("--- Verifying MultiHeadAttention Module ---")
# Create a dummy input tensor
dummy_config = DeepSeekMoEConfig()
dummy_input = torch.randint(0, dummy_config.vocab_size, (BATCH_SIZE, BLOCK_SIZE), device=DEVICE)
embedding_layer = nn.Embedding(dummy_config.vocab_size, dummy_config.n_embd, device=DEVICE)
dummy_x = embedding_layer(dummy_input)

# Instantiate and run the attention module
mha = MultiHeadAttention(dummy_config).to(DEVICE)
output = mha(dummy_x)

print(f"Input shape:  {dummy_x.shape}")
print(f"Output shape: {output.shape}")
assert dummy_x.shape == output.shape, "Output shape does not match input shape!"
print("MultiHeadAttention module implemented and verified successfully.")
print("=" * 60)

--- Verifying MultiHeadAttention Module ---
Input shape:  torch.Size([32, 256, 512])
Output shape: torch.Size([32, 256, 512])
MultiHeadAttention module implemented and verified successfully.


## Stage 2: Constructing Core Architectural Components

In this stage, we build the foundational components that both our Standard MoE and DeepSeek MoE models will use. The transformer architecture consists of:

1. **Multi-Head Attention**: Allows the model to attend to different positions in the input sequence
2. **Layer Normalization**: Stabilizes the activations between layers
3. **Feed-Forward Networks**: Processes token representations through MoE layers

### Multi-Head Attention Implementation

The Multi-Head Attention (MHA) mechanism is implemented following the standard approach from the "Attention is All You Need" paper:

1. Project input into query (Q), key (K), and value (V) representations
2. Split these projections across multiple attention heads
3. Compute scaled dot-product attention for each head
4. Concatenate outputs and project back to the original dimension

Our implementation includes optimizations like Flash Attention when available in PyTorch ≥ 2.0, which significantly improves performance for longer sequences.